In [1]:
import pandas as pd
import datetime as dt

# Load datasets
sales = pd.read_csv('sales.csv')
stores = pd.read_csv('stores.csv') # Columns: StoreID, Country, State, SquareMeters, OpenDate
products = pd.read_csv('products.csv')

In [6]:
# 1. Advanced Cleaning: Date Conversion
stores['OpenDate'] = pd.to_datetime(stores['Open Date'])
sales['Order Date'] = pd.to_datetime(sales['Order Date'])

In [7]:
# 2. Feature Engineering: Store Age
# Calculate how many years the store has been open relative to the last sale date
latest_date = sales['Order Date'].max()
stores['Store_Age_Years'] = (latest_date - stores['OpenDate']).dt.days / 365

In [9]:
# 3. Data Enrichment: Join Sales with Stores and Products
# This creates a "Master Denormalized Table" for SQL/Power BI
df = sales.merge(stores, on='StoreID').merge(products, on='ProductID')

In [17]:
# 4. Financial Calculations
# Convert columns to numeric data types first to handle any string values
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')  # Convert to numeric, invalid values become NaN
df['Unit Price USD'] = pd.to_numeric(df['Unit Price USD'], errors='coerce')  # Convert to numeric
df['Unit Cost USD'] = pd.to_numeric(df['Unit Cost USD'], errors='coerce')  # Fixed: corrected column name

# Now perform the calculations with numeric data
df['Revenue'] = df['Quantity'] * df['Unit Price USD']  # This will now work with numeric values
df['Total_Cost'] = df['Quantity'] * df['Unit Cost USD']  # Fixed: updated to use correct column name
df['Profit'] = df['Revenue'] - df['Total_Cost']

# Optional: Check for any missing values that might have been created during conversion
print("Any missing values after conversion:")
print(df[['Quantity', 'Unit Price USD', 'Unit Cost USD']].isnull().sum())  # Fixed: updated column name in check

Any missing values after conversion:
Quantity              1
Unit Price USD    62884
Unit Cost USD     62884
dtype: int64


In [19]:
# 5. Outlier Detection: Store Size vs Sales
# We flag stores that have unusually high square meters but low sales
q_size = df.groupby('StoreID')['Square Meters'].first().quantile(0.95)
print(f"Large Format Store Threshold: {q_size} sqm")


Large Format Store Threshold: 2100.0 sqm


In [21]:
# Save the enriched dataset
df.to_csv('Enriched_Sales_Data.csv', index=False)